# Optimizing model parameters

Everything so far was a piece of the loop: data in theme 2, transforms in theme 3, the model in theme 4, gradients in theme 5. This puts them together.

Training is a repeated guess and correct. The model guesses, the loss says how wrong it was, autograd computes the gradient, and the optimizer moves the weights against it.

One pass over the whole training set is an **epoch**.

In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

torch.manual_seed(42)

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(device)

## Data and model

Same FashionMNIST and the same 784-512-512-10 network as before.

In [ ]:
training_data = datasets.FashionMNIST(
    root="data", train=True, download=True, transform=ToTensor()
)
test_data = datasets.FashionMNIST(
    root="data", train=False, download=True, transform=ToTensor()
)

train_dataloader = DataLoader(training_data, batch_size=64, shuffle=True)
test_dataloader = DataLoader(test_data, batch_size=64, shuffle=False)

print(len(train_dataloader), len(test_dataloader))

In [ ]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28 * 28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),
        )

    def forward(self, x):
        return self.linear_relu_stack(self.flatten(x))


model = NeuralNetwork().to(device)

## Hyperparameters

Set by hand, not learned:

- **learning rate** is the step size. Too small and training crawls, too large and the loss jumps around or diverges.
- **batch size** is how many samples pass before one weight update.
- **epochs** is how many times the whole training set is used.

In [ ]:
learning_rate = 1e-3
epochs = 5

## Loss function

`nn.CrossEntropyLoss` is the standard choice for single-label classification with more than two classes.

It takes **logits**, not probabilities, because it applies log-softmax itself. That is why the model has no softmax layer.

It takes the target as an **int class index**, shape `[batch]`, not one-hot.

In [ ]:
loss_fn = nn.CrossEntropyLoss()

X, y = next(iter(train_dataloader))
X, y = X.to(device), y.to(device)
pred = model(X)

print(pred.shape, pred.dtype)
print(y.shape, y.dtype)
print(loss_fn(pred, y).item())

An untrained model on 10 classes should sit near `ln(10)`, about 2.30. A first loss far from that usually means something is wired wrong.

In [ ]:
import math

print(round(math.log(10), 4))

## Optimizer

The optimizer holds the parameters and the rule for updating them. `model.parameters()` is what connects it to the model, so anything not in that list will never change.

SGD is the simplest rule. Adam adapts the step size per parameter and usually needs less tuning.

In [ ]:
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

print(optimizer)

## The three steps

Inside every batch, in this order:

1. `optimizer.zero_grad()` clears the old gradients, because `backward()` adds to them instead of replacing them
2. `loss.backward()` computes the new gradients
3. `optimizer.step()` applies them to the weights

Order matters. Clearing after `backward()` throws away the gradients that were just computed, and the model never learns.

## train and eval mode

`model.train()` and `model.eval()` switch the behaviour of layers that act differently in the two phases, dropout and batch norm above all.

This network has neither, so here the calls change nothing. They are still written, because adding dropout later would silently break evaluation otherwise.

`torch.no_grad()` is a separate thing. `eval()` changes layer behaviour, `no_grad()` stops the graph being built. Evaluation wants both.

In [ ]:
def train_loop(dataloader, model, loss_fn, optimizer, device):
    size = len(dataloader.dataset)
    model.train()
    running_loss = 0.0

    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)

        pred = model(X)
        loss = loss_fn(pred, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * X.size(0)

        if batch % 300 == 0:
            seen = batch * dataloader.batch_size
            print(
                f"    batch {batch:>4d}  loss {loss.item():>7.4f}  [{seen:>5d}/{size}]"
            )

    return running_loss / size

In [ ]:
def test_loop(dataloader, model, loss_fn, device):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss = 0.0
    correct = 0

    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y).item()
            correct += (pred.argmax(1) == y).sum().item()

    return test_loss / num_batches, correct / size

`loss.item()` is used for logging on purpose. Keeping the tensor itself would keep its whole graph alive and leak memory across the epoch.

Accuracy compares `pred.argmax(1)` with the target. The prediction is a class index, so no softmax is needed. Softmax does not change which value is largest.

In [ ]:
for epoch in range(epochs):
    print(f"epoch {epoch + 1}")
    train_loss = train_loop(train_dataloader, model, loss_fn, optimizer, device)
    test_loss, accuracy = test_loop(test_dataloader, model, loss_fn, device)
    print(
        f"  train loss {train_loss:.4f}   test loss {test_loss:.4f}   "
        f"accuracy {accuracy:.4f}"
    )

## Reading the numbers

What to look at, in order:

- train loss should fall steadily. Flat from the start usually means the learning rate is too small, or `zero_grad` and `step` are in the wrong place.
- test loss should follow it down. When train loss keeps falling and test loss turns up, that is overfitting, which is theme 8.
- accuracy is the number to report, but loss is the number being optimized. They can move apart.

Plain SGD at `lr=1e-3` is deliberately slow here, so the trend is visible over a few epochs rather than jumping to a good score at once. Raising the learning rate or switching to Adam is the obvious next experiment.